# Notebook analyse de la saisonnalité Sensor P


In [2]:
# Importation des librairies
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf
import matplotlib.pyplot as plt

# Configuration de l'affichage par défaut
pd.options.plotting.backend = "plotly"

## 1. Chargement & Filtrage Stratégique

Nous avons observé une rupture brutale dans les données avant Avril 2025 (débits très faibles, ~50 m³/h). 
Pour cette analyse nous ne gardons que la période **post-Avril 2025**.

In [3]:
# 1. Chargement du fichier brut complet
file_path = 'bronze/bronze/sensor_P_20231110_20260106.csv'
df = pd.read_csv(file_path)

# 2. Pivot & Indexation temporelle
df['ts'] = pd.to_datetime(df['ts'])
df_pivot = df.pivot_table(index='ts', columns='sensor', values='value', aggfunc='first')

# 3. FILTRAGE TEMPOREL (La "Nouvelle Ère")
START_DATE = '2025-04-01'
df_recent = df_pivot[df_pivot.index >= START_DATE].copy()

# 4. Nettoyage Physique
# - ffill() : On comble les petits trous de transmission du capteur
# - clip(lower=0) : Un débit négatif est physiquement impossible, on ramène à 0.
df_recent = df_recent.ffill()
df_recent['debit_clean'] = df_recent['entry_debit_f1'].clip(lower=0)


## 2. Lissage Causal

Les données brutes montrent les cycles "Marche/Arrêt" de la pompe. Ce bruit technique masque l'information utile (combien d'eau arrive de la ville).

Donc on lisse avec center = False. ex: À midi, je fais la moyenne de 9h00 à 12h00. Je ne regarde que le passé. C'est ce que verra le modèle en temps réel.
Au lieu de center = True
À midi, je fais la moyenne de 10h30 à 13h30. mauvais pour prédiction, car à midi, je ne connais pas encore le débit de 13h30.

**Choix de la fenêtre : 3 Heures (180 min)** pour couvrir globalement les cycles longs de la pompe.

In [3]:
window_size = 180 # 3 Heures

# Application du lissage CAUSAL (Trailing window)
df_recent['debit_smooth_causal'] = df_recent['debit_clean'].rolling(window=window_size, center=False).mean()

# Suppression des NaN créés au tout début (les 3 premières heures)
df_analysis = df_recent.dropna(subset=['debit_smooth_causal']).copy()

# --- Visualisation de vérification ---
# On zoome sur 48h pour bien voir l'effet du lissage sur la pompe
df_zoom = df_analysis.iloc[2000:2000 + (60*48)] 

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_zoom.index, y=df_zoom['debit_clean'], 
                         name='Signal Brut (Pompe)', opacity=0.5, line=dict(color='orange', width=1)))
fig.add_trace(go.Scatter(x=df_zoom.index, y=df_zoom['debit_smooth_causal'], 
                         name='Signal Lissée (Apport Ville)', line=dict(color='green', width=3)))

fig.update_layout(title="Impact du Lissage Causal (3h) sur le signal brut", yaxis_title="Débit (m3/h)")
fig.show()

La courbe verte est un peu décalée vers la droite par rapport aux pics oranges. "retard" mathématique induit par le fait de ne regarder que le passé? Le modèle devra apprendre à gérer ce retard.

## 3. Analyse Statistique : Les Cycles de Vie de la Station

**Préparation :** Nous passons à une **maille horaire** (moyenne par heure). 

In [4]:
# Resampling Horaire
df_hourly = df_analysis[['debit_clean']].resample('1h').mean()
print("Données ré-échantillonnées à l'heure pour l'analyse statistique.")

Données ré-échantillonnées à l'heure pour l'analyse statistique.


### Autocorrélation (ACF)
**Question :** Est-ce que le débit d'aujourd'hui à 8h ressemble à celui d'hier à 8h ?

**Attente :** On s'attend à voir des pics de corrélation toutes les 24 heures.

In [ ]:
import plotly.graph_objects as go
from statsmodels.tsa.stattools import acf
import numpy as np

# alpha=0.05 nous donne l'intervalle de confiance (la zone bleue)
# nlags=350 correspond à environ 2 semaines d'heures
acf_values, confint = acf(df_hourly['debit_clean'].dropna(), nlags=350, alpha=0.05)

lags = np.arange(len(acf_values))

fig = go.Figure()

lower_bound = confint[:, 0] - acf_values
upper_bound = confint[:, 1] - acf_values

fig.add_trace(go.Scatter(
    x=lags, y=upper_bound,
    mode='lines', line=dict(width=0),
    name='Intervalle Confiance', hoverinfo='skip'
))
fig.add_trace(go.Scatter(
    x=lags, y=lower_bound,
    mode='lines', line=dict(width=0),
    fill='tonexty',
    fillcolor='rgba(0,0,255,0.2)',
    name='Intervalle Confiance', hoverinfo='skip'
))

fig.add_trace(go.Bar(
    x=lags, y=acf_values,
    width=0.2,
    marker_color='rgb(31, 119, 180)',
    name='Corrélation'
))
fig.add_trace(go.Scatter(
    x=lags, y=acf_values,
    mode='markers',
    marker=dict(color='rgb(31, 119, 180)', size=6),
    showlegend=False,
    name='Corrélation'
))


ticks_vals = np.arange(0, 351, 24)
ticks_text = [f"{i}h ({(i//24)}j)" if i > 0 else "0h" for i in ticks_vals]

fig.update_layout(
    title="Autocorrélogramme Interactif (Preuve des cycles de 24h)",
    xaxis_title="Lag (Retard en Heures)",
    yaxis_title="Autocorrélation (Pearson)",
    yaxis=dict(range=[-1, 1]),
    xaxis=dict(
        tickmode='array',
        tickvals=ticks_vals,
        ticktext=ticks_text,
        gridcolor='lightgray' 
    ),
    height=500
)

fig.add_hline(y=0, line_color="black", line_width=1)

fig.show()

* Les pics réguliers montrent que le phénomène est fortement cyclique.
* Le débit à l'instant T est fortement corrélé au débit d'il y a exactement 24h, 48h, etc. Cela valide statistiquement l'existence d'un cycle journalier lié à l'activité humaine --> utiliser l'heure de la journée comme feature dans notre modèle

### B. Preuve n°2 : Décomposition du Cycle Journalier (24h)
Nous utilisons l'algorithme STL (*Seasonal-Trend-Loess*) avec une **période de 24**.

In [ ]:
res_daily = seasonal_decompose(df_hourly['debit_clean'].dropna(), model='additive', period=24)

def plot_decomposition_interactive(res_object, period_name):
    fig_trend = go.Figure()
    fig_trend.add_trace(go.Scatter(x=res_object.trend.index, y=res_object.trend, line=dict(color='red', width=2), name='Tendance'))
    fig_trend.update_layout(title=f"Tendance de fond ({period_name}) - Inclut Phénomènes de ressuyage par ex & Saisons longues (semaine, mensuelles...)", height=300)
    fig_trend.show()
    
    fig_seas = go.Figure()
    fig_seas.add_trace(go.Scatter(x=res_object.seasonal.index, y=res_object.seasonal, line=dict(color='green', width=1.5), name='Saisonnalité'))
    fig_seas.update_layout(title=f"Saisonnalité Pure ({period_name})", height=300)
    fig_seas.show()
    
    # 3. Residuals
    fig_resid = go.Figure()
    fig_resid.add_trace(go.Scatter(x=res_object.resid.index, y=res_object.resid, mode='markers', marker=dict(color='blue', size=2, opacity=0.5), name='Résidus'))
    fig_resid.add_hline(y=0, line_dash="dash", line_color="gray")
    fig_resid.update_layout(title=f"Résidus ({period_name}) - Anomalies & Bruit", height=300)
    fig_resid.show()

# Affichage
plot_decomposition_interactive(res_daily, "Cycle 24h")

**Saisonnalité 24h :**
En zoomant sur quelques jours:
* Pic à 11h
* Petit pic à 14H
* Creux de milieu de journée à 16H
* Pic le soir 18h-20H
* Creux la nuit autour de 23H - 6H

### C. Preuve n°3 : Décomposition du Cycle Hebdomadaire (7 Jours)
L'algorithme précédent ne pouvait voir qu'un seul cycle à la fois. Relançons-le avec une **période de 168** (24h * 7j) pour voir si le week-end est différent.

In [ ]:
res_daily = seasonal_decompose(df_hourly['debit_clean'].dropna(), model='additive', period=168)

def plot_decomposition_interactive(res_object, period_name):
    fig_trend = go.Figure()
    fig_trend.add_trace(go.Scatter(x=res_object.trend.index, y=res_object.trend, line=dict(color='red', width=2), name='Tendance'))
    fig_trend.update_layout(title=f"Tendance de fond ({period_name}) - Inclut Phénomènes de ressuyage par ex & Saisons longues", height=300)
    fig_trend.show()
    
    fig_seas = go.Figure()
    fig_seas.add_trace(go.Scatter(x=res_object.seasonal.index, y=res_object.seasonal, line=dict(color='green', width=1.5), name='Saisonnalité'))
    fig_seas.update_layout(title=f"Saisonnalité Pure ({period_name})", height=300)
    fig_seas.show()
    
    fig_resid = go.Figure()
    fig_resid.add_trace(go.Scatter(x=res_object.resid.index, y=res_object.resid, mode='markers', marker=dict(color='blue', size=2, opacity=0.5), name='Résidus'))
    fig_resid.add_hline(y=0, line_dash="dash", line_color="gray")
    fig_resid.update_layout(title=f"Résidus ({period_name}) - Anomalies & Bruit", height=300)
    fig_resid.show()

plot_decomposition_interactive(res_daily, "Cycle 7j")

Tendance : On voit une diminution du débit tout le mois d'août : vacances? temps sec?

->> On voit des pics commun entre les résidus et la tendance: ex 26 juin. Ressuyage? début septembre : rentrée?

->> feature booléenne pour dire si c'est les vacances? feature de cumul de pluie tombée? et pour sûr feature par mois

Saisonnalité:

On voit des pics en fin de semaine: ex Samedi 5 avril, dimanche 6 avril 2025

-> feature du jour de la semaine, et éventuellement feature booléenne pour le wk?

In [ ]:
import plotly.express as px
import pandas as pd

df_plot = df_hourly.copy()
df_plot['day_name'] = df_plot.index.day_name()
df_plot['hour'] = df_plot.index.hour
df_plot['is_weekend'] = df_plot.index.dayofweek >= 5 

days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig_box = px.box(df_plot, x='day_name', y='debit_clean',
                 category_orders={'day_name': days_order},
                 color='is_weekend', 
                 title="Distribution du Débit par Jour (Est-ce que le Dimanche est plus haut ?)",
                 labels={'debit_clean': 'Débit (m3/h)', 'day_name': 'Jour', 'is_weekend': 'C\'est le Week-end ?'},
                 color_discrete_map={False: 'blue', True: 'red'})

fig_box.show()

df_profile = df_plot.groupby(['hour', 'is_weekend'])['debit_clean'].mean().reset_index()

df_profile['Type de jour'] = df_profile['is_weekend'].map({False: 'Semaine (Lun-Ven)', True: 'Week-end (Sam-Dim)'})

fig_profile = px.line(df_profile, x='hour', y='debit_clean', color='Type de jour',
                      title="Comparaison des Profils Horaires",
                      labels={'debit_clean': 'Débit Moyen (m3/h)', 'hour': 'Heure de la journée'},
                      color_discrete_map={'Semaine (Lun-Ven)': 'blue', 'Week-end (Sam-Dim)': 'red'})

fig_profile.update_traces(mode='lines+markers')
fig_profile.show()

Le comportement change le week end!

## 4. Analyse Macroscopique : Tendances Mensuelles
Enfin, regardons l'évolution globale mois par mois pour déceler une éventuelle saisonnalité annuelle (Hiver vs Été).

In [ ]:
import plotly.express as px

# 1. CORRECTION : On utilise 'MS' (Month Start) au lieu de 'ME'
df_monthly = df_hourly.resample('MS').mean().reset_index()

df_monthly['Mois_Label'] = df_monthly['ts'].dt.strftime('%B %Y') # Ex: "April 2025"

fig_monthly = px.bar(df_monthly, x='Mois_Label', y='debit_clean',
                     title="Débit Moyen Mensuel (Avril 2025 - Jan 2026)",
                     labels={'debit_clean': 'Débit Moyen (m3/h)', 'Mois_Label': 'Mois'},
                     text_auto='.0f',
                     color='debit_clean', 
                     color_continuous_scale='Teal')

fig_monthly.update_xaxes(type='category') 

fig_monthly.show()

-> débit élevé au printemps (eau de l'automne/hiver?), faible en été et remontée en automne.
feature mois

In [ ]:
df_analysis.to_csv('bronze/bronze/sensor_P_analysis_ready.csv')
print(" Fichier 'sensor_P_analysis_ready.csv' sauvegardé avec succès.")